[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/langgraph/langgraph_promptchaining.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# LangGraph：提示词链

本 Notebook 演示如何使用 LangGraph 提示词链构建多步骤邮件助手。助手将写作过程分为三个阶段：

- 根据主题和要点生成提纲

- 根据提纲与目标语气撰写初稿

- 按需调整语气，使其符合指定风格

将任务拆解为逻辑步骤，可以精细控制内容生成过程。图中每个阶段由独立节点负责，支持有针对性的提示词、中间输出和条件逻辑。

整个工作流还接入了 Phoenix 插桩，获得基于 OpenTelemetry 的追踪与调试能力。你可以直接在 Phoenix 界面查看每一步的输入、输出和转移，定位生成过程中的瓶颈或偏差。

In [ ]:
!pip install langgraph langchain langchain_community "arize-phoenix" arize-phoenix-otel openinference-instrumentation-langchain langchain-openai

这是一个 LangGraph 提示词链模板，实现邮件撰写的三个步骤：编写提纲、撰写邮件、调整语气。

In [ ]:
import os
from getpass import getpass

from langgraph.graph import END, START, StateGraph

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

# 配置 Phoenix 追踪
请前往 https://app.phoenix.arize.com/ 生成 API 密钥，以便使用 Phoenix 追踪 LangGraph 应用。

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="提示词链", auto_instrument=True)

# 选择 LLM

In [ ]:
from langchain_openai import ChatOpenAI
from typing_extensions import Literal, TypedDict

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# 定义图状态

*EmailState* 定义邮件写作智能体的共享内存。每个字段代表邮件逐步变化的状态：从用户的初始输入（主题、笔记、语气），到提纲生成、草稿撰写以及最终语气调整。状态字典在节点之间传递，确保工作流持续保留上下文并逐步更新。

In [ ]:
class EmailState(TypedDict, total=False):
    subject: str
    bullet_points: str  # 用户原始笔记
    desired_tone: str  # "formal"、"friendly" 等。
    outline: str  # 节点 1 的结果
    draft_email: str  # 节点 2 的结果
    final_email: str  # 语气调整后的结果（如有需要）

# 分步提示词链：提纲 → 草稿 → 语气检查

此工作流串联多次 LLM 调用，将原始笔记转化为完善的邮件：

**generate_outline**：将用户要点转换为结构化提纲。

**write_email**：按目标语气将提纲扩展为完整邮件草稿。

**tone_gate**：通过轻量 LLM 分类检查草稿是否符合所需语气。

**reform_tone**：语气不符时，在保留内容的前提下重写草稿。

各节点相互独立，便于针对性调试，也可在不同任务或格式中复用。这种多步骤打磨方式类似人工写作过程，有助于提高输出质量。

In [ ]:
def generate_outline(state: EmailState) -> EmailState:
    """LLM 调用 1：根据要点生成提纲。"""
    prompt = (
        "为邮件创建简洁提纲。\n"
        f"主题： {state['subject']}\n"
        f"要点：\n{state['bullet_points']}\n"
        "以编号要点返回提纲。"
    )
    outline = llm.invoke(prompt).content
    return {"outline": outline}


def write_email(state: EmailState) -> EmailState:
    """LLM 调用 2：根据提纲撰写邮件。"""
    prompt = (
        f"根据以下提纲撰写完整邮件：\n{state['outline']}\n\n"
        f"语气： {state['desired_tone']}\n"
        "以问候开头，采用专业格式，保持简洁。"
    )
    email = llm.invoke(prompt).content
    return {"draft_email": email}


def tone_gate(state: EmailState) -> Literal["Pass", "Fail"]:
    """
    门控：快速启发式判断：
      Pass → 邮件已包含所需语气关键词。
      Fail → 其他情况（再调用一次 LLM 调整）。
    """
    prompt = (
        f"检查以下邮件是否符合目标语气 {state['desired_tone']}:\n\n"
        f"{state['draft_email']}\n"
        f"若符合，返回 'Pass'；否则返回 'Fail'。"
    )
    return llm.invoke(prompt).content.strip()


def reform_tone(state: EmailState) -> EmailState:
    """LLM 调用 3：重写邮件以符合目标语气。"""
    prompt = (
        f"重写以下邮件，使其具有 {state['desired_tone']} 语气。\n\n"
        f"邮件：\n{state['draft_email']}\n\n"
        "保持内容一致，但调整表达、用词与落款。"
    )
    final_email = llm.invoke(prompt).content
    return {"final_email": final_email}

# 使用 LangGraph 编译邮件提示词链

这里组装完整的邮件生成 LangGraph。图从 outline_generator 开始，进入 email_writer，仅当语气检查失败时才路由到 tone_reformer。这展示了根据模型输出动态调整控制流的提示词链模式。编译后的图可以接收用户输入，并通过 Phoenix 追踪以进行调试或优化。

In [ ]:
graph = StateGraph(EmailState)

graph.add_node("outline_generator", generate_outline)
graph.add_node("email_writer", write_email)
graph.add_node("tone_reformer", reform_tone)

# 边
graph.add_edge(START, "outline_generator")
graph.add_edge("outline_generator", "email_writer")
graph.add_conditional_edges(
    "email_writer",
    tone_gate,
    {"Pass": END, "Fail": "tone_reformer"},
)
graph.add_edge("tone_reformer", END)

email_chain = graph.compile()

# ────────────────────────────────────────────────
# 4. 可视化并运行一次
# ────────────────────────────────────────────────
# display(Image(email_chain.get_graph().draw_mermaid_png()))

# 使用示例

In [ ]:
initial_state = email_chain.invoke(
    {
        "subject": "Quarterly Sales Recap & Next Steps",
        "bullet_points": "- Q1 revenue up 18%\n- Need feedback on new pricing tiers\n- Reminder: submit pipeline forecasts by Friday",
        "desired_tone": "friendly",
    }
)

print("\n========== EMAIL ==========")
print(initial_state.get("final_email", initial_state["draft_email"]))

# 记得在 Phoenix 中查看追踪！

# 添加评估（Evals）

本节评估各智能体完成其任务的情况。

- 提纲生成：评估清晰度、结构与相关性。
- 邮件撰写：评估语法、拼写和内容连贯性。
- 语气检查：评估语气检测的通过或失败判断是否正确。
- 语气调整：评估重写后的邮件是否符合目标语气。

In [ ]:
OUTLINE_EVAL_TEMPLATE = """
你正在评估用于撰写邮件的提纲。

邮件主题：{subject}
要点：{bullet_points}

生成的提纲：
{outline}

请按以下标准评估：
1. 清晰度与结构：是否组织合理、易于理解？
2. 相关性：是否完整反映提供的要点？

返回以下之一：
- 0/2：两项均差
- 1/2：仅一项良好
- 2/2：两项均良好
"""

EMAIL_EVAL_TEMPLATE = """
你正在评估邮件质量。

使用的提纲：
{outline}

邮件草稿：
{final_email}

按以下标准评估：
1. 语法与拼写：是否存在明显错误？
2. 内容连贯性：是否遵循提纲的结构与含义？

返回以下之一：
- 0/2：两项均差
- 1/2：仅一项良好
- 2/2：两项均良好
"""

TONE_REFORM_EVAL_TEMPLATE = """
你正在评估最终邮件的语气是否符合目标语气。

目标语气：{desired_tone}

最终邮件：
{final_email}

最终邮件是否符合目标语气？

返回 "yes" 或 "no"
"""

# 从 Phoenix 获取跨度

In [ ]:
from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery

df = Client().spans.get_spans_dataframe(
    query=SpanQuery().where("name == 'LangGraph'"), project_name="提示词链"
)

In [ ]:
df.to_csv("prompt_chaining_spans")

# 邮件评估的自定义设置

In [ ]:
import json

import pandas as pd
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe


def unpack(row):
    blob = json.loads(row["attributes.output.value"])
    return pd.Series(
        {
            "subject": blob.get("subject"),
            "bullet_points": blob.get("bullet_points"),
            "outline": blob.get("outline"),
            "desired_tone": blob.get("desired_tone"),
            "draft_email": blob.get("draft_email"),
            "final_email_raw": blob.get("final_email"),  # 可能为 None / ""
        }
    )


df = df.join(df.apply(unpack, axis=1))


def pick_email(row):
    fe = row["final_email_raw"]
    if fe and fe.strip():
        return fe
    return row["draft_email"]


df["final_email"] = df.apply(pick_email, axis=1)
df

# 生成评估

In [ ]:
llm = LLM(provider="openai", model="gpt-4o")

outline_evaluator = ClassificationEvaluator(
    name="outline_eval",
    llm=llm,
    prompt_template=OUTLINE_EVAL_TEMPLATE,
    choices={"0/2": 0.0, "1/2": 0.5, "2/2": 1.0},
)

email_evaluator = ClassificationEvaluator(
    name="email_eval",
    llm=llm,
    prompt_template=EMAIL_EVAL_TEMPLATE,
    choices={"0/2": 0.0, "1/2": 0.5, "2/2": 1.0},
)

tone_evaluator = ClassificationEvaluator(
    name="tone_eval",
    llm=llm,
    prompt_template=TONE_REFORM_EVAL_TEMPLATE,
    choices={"yes": 1.0, "no": 0.0},
)

outline_results = await async_evaluate_dataframe(
    dataframe=df.dropna(subset=["outline"]),
    evaluators=[outline_evaluator],
    concurrency=10,
)

# email_evaluator 与 tone_evaluator 使用相同子集，因此合并处理
email_and_tone_results = await async_evaluate_dataframe(
    dataframe=df.dropna(subset=["final_email"]),
    evaluators=[email_evaluator, tone_evaluator],
    concurrency=10,
)

# 将评估导出到 Phoenix！

In [ ]:
from phoenix.client import AsyncClient
from phoenix.evals.utils import to_annotation_dataframe

px_client = AsyncClient()

# 将提纲结果转换为标注格式并上传
outline_annotations = to_annotation_dataframe(outline_results, ["outline_eval"])
await px_client.spans.log_span_annotations_dataframe(
    dataframe=outline_annotations,
    annotator_kind="LLM",
)

# 将邮件与语气结果转换为标注格式并上传
email_and_tone_annotations = to_annotation_dataframe(
    email_and_tone_results, ["email_eval", "tone_eval"]
)
await px_client.spans.log_span_annotations_dataframe(
    dataframe=email_and_tone_annotations,
    annotator_kind="LLM",
)